## Without web search

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=model
    )

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [5]:
print(response['messages'][-1].content)

Short answer: My training data goes up to June 2024. I don’t have knowledge of events or developments that happened after that unless you share details in the chat.

Notes:
- I don’t automatically browse the web unless your platform has browsing enabled.
- I can still help with general knowledge, concepts, problem solving, coding, writing, and reasoning.
- If you need recent info, you can provide a link or summary, and I can help interpret or compare it.

What would you like to know? If you have a specific topic in mind, I can tailor the answer to that.


## Add web search tool

In [6]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'There have been 45 individuals who have served as mayor in San Francisco since 1850, when California became a state following the American Conquest of California. Prior to the conquest, Californios served as Mayor of San Francisco during the Spanish and Mexican eras since 1779.\n\nThe current mayor is Democrat Daniel Lurie.\n\n## Elections\n\n[edit]\n\nMain articles: 1979 San Francisco mayoral election, 1995, 1999, 2003, 2007, 2011, 2015, 2018, 2019, and 2024 [...] ## Salary and benefits\n\n[edit]\n\nAs of 2024, the mayor is paid an annual salary of $364,582, the highest mayoral salary in the United States. Nine city public employees earned higher salaries than the mayor, including the chief investment officer and the managing dir

In [19]:
agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [20]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='72409301-01a8-46c3-9105-e19ee6df5423'),
 AIMessage(content='', additional_kwargs={'reasoning_details': [{'data': 'gAAAAABqix8-8j723WKR50AA91GWFX9NgrFWv-KhQY9ubXJ-V-URUPc4ASOuzjEaPiOt8nwXUpR6qdbMSaEVRKRkOpQmH5vF5pCGNvYP6UDQ7XCc8yZfY-pKVDwi7CrHTtb0y6UG1l7-Bv3p-U8eeRCBzXCUxNUriBUI6LJsAkzVhuR0YngENX3_C19JDpM6Pr14hwbAdg200vipUaox7mwiqxyh6RDfTiiZFwwW9WqYYFlgsxp90M7i6F-hfLyegsizTlIV_QAlSVW8B8fhGokcMUqPdwMErAyo4_C4tpHAlp-UFir8qtsyIuj3ujx9Ym59eqPNC6KvE6ymzrg7npmmxK-Guu5sGKRmIOeCwgMCQwzAUK9YIV9PR5yayRru4fpDb69X_Rrf-PeDqsAFrO0tCUcVWrHtqGehgj_B2uy5Y4eTn9Ve8e_4KLGCSaOUuzzFnMwmdQWc5SdOZB3Jv_-N6WL_6lfxMX3MesX_dWDeuBT7w04Hu4LBsv5a2Y37-2bNE_lvNAhGScIqjUx7vPJm-OUj5eaV_2_H8llQ49lFJ0XxGIAhERXNXRZCBVNnLAS8l7MY1QAyPF-RVRA37PA-L8I8mqfAvYxarSrpq9Jw5TphoflhlHLS9wuv9JJEyFDntYIebZdwOu0Vj7XF2Q_rL-zaHWfZAZz40rKfzg-p8YwHiIEIV4KSRmM58U3Y7mxCfgyRjoB6IviXMtjkLV1qN6Pp2M9WhDdYlNj2jQXVhORfMqKckOcOQLFldxstrw

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r